# Lab 4 — RAG e bases de conhecimento

Neste notebook, você irá construir um mecanismo de Retrieval-Augmented Generation (RAG) para que a NIA responda usando documentos fictícios da TechStore.

## Objetivos

- entender o papel dos embeddings;
- criar um pequeno índice vetorial;
- recuperar trechos por similaridade semântica;
- gerar respostas fundamentadas;
- indicar fontes e reconhecer ausência de informação.

> Todo o conteúdo da TechStore neste laboratório é fictício e possui finalidade acadêmica.

## 0. Antes de começar

Use a mesma configuração de chave dos laboratórios anteriores:

1. Abra **Secrets** na barra lateral do Colab.
2. Crie ou reutilize o secret `GEMINI_API_KEY`.
3. Habilite o acesso para este notebook.

> Nunca escreva a chave diretamente no código compartilhado.

## 1. Instale o SDK

O NumPy será usado apenas para calcular a similaridade entre vetores.

In [ ]:
!pip install -q -U "google-genai>=2.3.0" "numpy>=1.26"

## 2. Crie o cliente e escolha os modelos

Usaremos um modelo para gerar respostas e outro para criar embeddings.

In [ ]:
from google import genai
from google.colab import userdata
import numpy as np

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash"
EMBEDDING_MODEL = "gemini-embedding-2"

print("Cliente criado com sucesso.")

## 3. Crie a base de conhecimento

Cada item abaixo representa um trecho de um documento. Em sistemas reais, os documentos costumam ser divididos em chunks menores e armazenados com metadados como título, versão, data e URL.

Para manter o notebook independente, nossa pequena base ficará em memória.

In [ ]:
DOCUMENTOS = [
    {
        "fonte": "politica-garantia.md",
        "titulo": "Garantia de produtos",
        "conteudo": (
            "Os produtos da TechStore usados neste laboratório possuem garantia fictícia "
            "de 12 meses contra defeitos de fabricação. Danos por queda, contato com "
            "líquidos e uso inadequado não são cobertos."
        ),
    },
    {
        "fonte": "politica-devolucao.md",
        "titulo": "Devoluções",
        "conteudo": (
            "Uma compra fictícia realizada pela internet pode ter a devolução solicitada "
            "em até 7 dias corridos após o recebimento. O produto deve ser devolvido com "
            "seus acessórios."
        ),
    },
    {
        "fonte": "atendimento.md",
        "titulo": "Atendimento humano",
        "conteudo": (
            "O suporte humano funciona de segunda a sexta, das 9h às 18h, no fuso "
            "America/Sao_Paulo. Não há atendimento humano aos sábados e domingos."
        ),
    },
    {
        "fonte": "entregas.md",
        "titulo": "Acompanhamento de pedidos",
        "conteudo": (
            "A NIA não possui acesso ao sistema de pedidos ou ao rastreamento de entregas. "
            "Para consultar uma entrega, o cliente deve usar a área Meus Pedidos com o "
            "número do pedido."
        ),
    },
    {
        "fonte": "privacidade.md",
        "titulo": "Dados sensíveis",
        "conteudo": (
            "O suporte nunca solicita senha, código de autenticação ou número completo de "
            "cartão. Caso uma dessas informações seja enviada, ela não deve ser repetida "
            "na resposta nem registrada em chamados."
        ),
    },
    {
        "fonte": "manual-orion-14.md",
        "titulo": "Orion 14 — problema de energia",
        "conteudo": (
            "Se o notebook Orion 14 não ligar, desconecte acessórios, verifique a tomada "
            "e mantenha o botão de energia pressionado por 10 segundos. Se não houver luz "
            "indicadora, procure o suporte técnico."
        ),
    },
    {
        "fonte": "manual-orion-14.md",
        "titulo": "Orion 14 — conexão sem fio",
        "conteudo": (
            "Para falhas de Wi-Fi no Orion 14, desative e reative o modo avião, reinicie o "
            "roteador e teste outra rede. A redefinição completa de rede deve ser feita "
            "somente com orientação do suporte."
        ),
    },
    {
        "fonte": "catalogo.md",
        "titulo": "Acessórios do Orion 14",
        "conteudo": (
            "O Orion 14 utiliza carregador USB-C fictício de 65 W. O catálogo deste "
            "laboratório não informa preços nem disponibilidade de estoque em tempo real."
        ),
    },
]

print(f"Trechos carregados: {len(DOCUMENTOS)}")
for documento in DOCUMENTOS:
    print("-", documento["fonte"], "|", documento["titulo"])

## 4. Transforme texto em vetor

Um embedding representa características do conteúdo como uma lista de números. Primeiro, gere o vetor de uma única frase e inspecione apenas seu tamanho e seus primeiros valores.

In [ ]:
exemplo = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents="task: sentence similarity | query: notebook sem energia",
)

vetor_exemplo = np.array(exemplo.embeddings[0].values, dtype=float)
print("Dimensões:", vetor_exemplo.shape)
print("Primeiros valores:", vetor_exemplo[:8])

### Prepare perguntas e documentos

Busca é um caso assimétrico: uma pergunta e um documento exercem papéis diferentes. Por isso, incluiremos instruções apropriadas no texto enviado ao modelo de embeddings.

In [ ]:
def preparar_pergunta(pergunta: str) -> str:
    return f"task: question answering | query: {pergunta}"


def preparar_documento(titulo: str, conteudo: str) -> str:
    return f"title: {titulo} | text: {conteudo}"


print(preparar_pergunta("Como aciono a garantia?"))
print(preparar_documento(DOCUMENTOS[0]["titulo"], DOCUMENTOS[0]["conteudo"]))

## 5. Crie o índice vetorial

Geraremos um embedding separado para cada trecho. O resultado ficará em memória junto com sua fonte.

> Esta etapa faz uma chamada de API por trecho. Em produção, seriam considerados processamento em lote, cache persistente e atualização incremental.

In [ ]:
def gerar_embedding(texto: str) -> np.ndarray:
    resultado = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texto,
    )
    return np.array(resultado.embeddings[0].values, dtype=float)


INDICE = []
for documento in DOCUMENTOS:
    texto_preparado = preparar_documento(
        documento["titulo"], documento["conteudo"]
    )
    item = documento.copy()
    item["embedding"] = gerar_embedding(texto_preparado)
    INDICE.append(item)

print(f"Índice criado com {len(INDICE)} vetores.")
print("Dimensões de cada vetor:", INDICE[0]["embedding"].shape)

## 6. Calcule a similaridade de cosseno

A similaridade compara a direção de dois vetores. Quanto maior a pontuação, maior a proximidade no espaço de embeddings.

In [ ]:
def similaridade_cosseno(vetor_a: np.ndarray, vetor_b: np.ndarray) -> float:
    denominador = np.linalg.norm(vetor_a) * np.linalg.norm(vetor_b)
    if denominador == 0:
        return 0.0
    return float(np.dot(vetor_a, vetor_b) / denominador)


print("Vetor consigo mesmo:", similaridade_cosseno(vetor_exemplo, vetor_exemplo))

## 7. Busque os trechos mais próximos

A função cria o embedding da pergunta, compara-o com todos os documentos e devolve os `k` itens com maior pontuação.

In [ ]:
def buscar_trechos(pergunta: str, k: int = 3) -> list[dict]:
    vetor_pergunta = gerar_embedding(preparar_pergunta(pergunta))
    resultados = []

    for item in INDICE:
        resultados.append(
            {
                "fonte": item["fonte"],
                "titulo": item["titulo"],
                "conteudo": item["conteudo"],
                "similaridade": similaridade_cosseno(
                    vetor_pergunta, item["embedding"]
                ),
            }
        )

    resultados.sort(key=lambda item: item["similaridade"], reverse=True)
    return resultados[:k]


pergunta = "Meu Orion não dá nenhum sinal quando aperto o botão. O que verifico?"
trechos = buscar_trechos(pergunta)

for posicao, trecho in enumerate(trechos, start=1):
    print(f"{posicao}. {trecho['similaridade']:.4f} | {trecho['fonte']}")
    print(trecho["conteudo"])
    print()

### Pare e observe

- O primeiro trecho realmente ajuda a responder?
- Os demais são relevantes ou apenas relacionados ao tema?
- A pontuação informa que o conteúdo é verdadeiro?
- O que mudaria se `k` fosse 1 ou 5?

Similaridade é um sinal de proximidade, não uma prova de resposta correta.

## 8. Compare com uma resposta sem RAG

Primeiro, peça ao modelo que responda sem fornecer os documentos. A resposta pode parecer plausível, mas não sabemos em qual política ela se baseou.

In [ ]:
pergunta_comparacao = "Por quantos meses vale a garantia e o que ela não cobre?"

sem_rag = client.interactions.create(
    model=MODEL,
    input=(
        "Responda como assistente da TechStore fictícia: "
        f"{pergunta_comparacao}"
    ),
)

print("RESPOSTA SEM RAG:\n")
print(sem_rag.output_text)

## 9. Monte o contexto recuperado

Agora a aplicação irá buscar os trechos e formatá-los com a fonte. Esse texto será enviado junto com a pergunta.

In [ ]:
def montar_contexto(trechos: list[dict]) -> str:
    blocos = []
    for trecho in trechos:
        blocos.append(
            f"FONTE: [{trecho['fonte']}]\n"
            f"TÍTULO: {trecho['titulo']}\n"
            f"CONTEÚDO: {trecho['conteudo']}"
        )
    return "\n\n---\n\n".join(blocos)


trechos_comparacao = buscar_trechos(pergunta_comparacao, k=3)
contexto_comparacao = montar_contexto(trechos_comparacao)
print(contexto_comparacao)

## 10. Gere uma resposta fundamentada

O System Prompt limita a resposta ao contexto recuperado e define como citar as fontes. Quando o contexto não sustentar a resposta, a NIA deve reconhecer a ausência da informação.

In [ ]:
SYSTEM_PROMPT_RAG = "\n".join([
    "Você é a NIA, assistente da TechStore fictícia.",
    "Responda usando apenas o CONTEXTO fornecido pela aplicação.",
    "Não use conhecimento próprio para completar informações ausentes.",
    "Se o contexto não contiver a resposta, diga claramente: "
    "'Não encontrei essa informação na base da TechStore.'",
    "Não invente políticas, produtos, preços, prazos ou procedimentos.",
    "Ao final de cada afirmação factual, indique a fonte no formato [arquivo.md].",
    "Seja objetiva, responda em português e use no máximo 150 palavras.",
])


com_rag = client.interactions.create(
    model=MODEL,
    system_instruction=SYSTEM_PROMPT_RAG,
    input=(
        f"CONTEXTO:\n{contexto_comparacao}\n\n"
        f"PERGUNTA:\n{pergunta_comparacao}"
    ),
)

print("RESPOSTA COM RAG:\n")
print(com_rag.output_text)

### Compare as duas respostas

Registre:

- Qual resposta é mais fácil de verificar?
- A resposta sem RAG afirmou algo diferente dos documentos?
- A resposta com RAG citou uma fonte que realmente sustenta a afirmação?
- Os trechos recuperados continham informação desnecessária?

## 11. Reúna o pipeline em uma função

O pipeline completo executa duas etapas diferentes:

1. **recuperação:** pergunta → embedding → trechos;
2. **geração:** pergunta + trechos → resposta.

Manter essas etapas separadas facilita inspecionar e testar o sistema.

In [ ]:
def responder_com_rag(
    pergunta: str,
    k: int = 3,
    mostrar_trechos: bool = True,
) -> str:
    trechos = buscar_trechos(pergunta, k=k)

    if mostrar_trechos:
        print("[Trechos recuperados]")
        for trecho in trechos:
            print(
                f"- {trecho['similaridade']:.4f} | "
                f"{trecho['fonte']} | {trecho['titulo']}"
            )
        print()

    contexto = montar_contexto(trechos)
    interaction = client.interactions.create(
        model=MODEL,
        system_instruction=SYSTEM_PROMPT_RAG,
        input=(
            f"CONTEXTO:\n{contexto}\n\n"
            f"PERGUNTA:\n{pergunta}"
        ),
    )
    return interaction.output_text

### Faça uma consulta completa

In [ ]:
resposta = responder_com_rag(
    "Comprei pela internet e me arrependi. Até quando posso pedir a devolução?"
)
print("NIA:", resposta)

## 12. Teste uma informação ausente

A base não contém condições de parcelamento. Os trechos mais próximos ainda serão recuperados, mas eles não devem ser usados para inventar uma resposta.

In [ ]:
resposta_ausente = responder_com_rag(
    "Posso parcelar um notebook em 24 vezes sem juros?"
)
print("NIA:", resposta_ausente)

!!! important

    Recuperar sempre os itens mais próximos não significa que algum deles contém a resposta. Em sistemas reais, podem ser usados limiares, reranking, filtros de metadados e avaliações. Nenhum valor de similaridade é universal para todos os modelos e bases.

## 13. Testes obrigatórios

Execute e avalie os quatro casos.

| Teste | Pergunta | O que observar |
| --- | --- | --- |
| Informação direta | `Qual é o horário do suporte?` | O trecho e a fonte estão corretos? |
| Paráfrase | Pergunta sobre desistência sem usar “devolução” | A busca semântica encontra a política? |
| Informação ausente | Pergunta sobre financiamento | A NIA reconhece a ausência? |
| Ambiguidade | `Qual é o prazo?` | O sistema pede contexto ou inventa um prazo? |

In [ ]:
testes = [
    "Qual é o horário do suporte e ele funciona aos domingos?",
    "Recebi a compra ontem, mas mudei de ideia. Ainda posso desistir?",
    "Quais são as opções de financiamento da TechStore?",
    "Qual é o prazo?",
]

for numero, pergunta_teste in enumerate(testes, start=1):
    print(f"\n--- TESTE {numero} ---")
    print("Você:", pergunta_teste)
    print("NIA:", responder_com_rag(pergunta_teste))

### Registro dos testes

**Teste 1 — Informação direta**

Trecho recuperado e resultado:

**Teste 2 — Paráfrase**

Trecho recuperado e resultado:

**Teste 3 — Informação ausente**

A NIA reconheceu a ausência? Alguma fonte foi citada indevidamente?

**Teste 4 — Ambiguidade**

A NIA pediu esclarecimento ou associou a pergunta a um prazo específico?

**Falha ou limitação observada:**

## 14. Avalie a recuperação separadamente

Uma resposta ruim pode ter pelo menos duas causas:

- o trecho correto não foi recuperado;
- o trecho estava presente, mas o modelo o utilizou incorretamente.

Antes de alterar o prompt, verifique se o documento esperado aparece entre os primeiros resultados.

In [ ]:
casos_esperados = [
    ("Quanto tempo dura a cobertura contra defeitos?", "politica-garantia.md"),
    ("Meu Orion não encontra a rede sem fio.", "manual-orion-14.md"),
    ("A assistente pode pedir minha senha?", "privacidade.md"),
]

acertos = 0
for pergunta_teste, fonte_esperada in casos_esperados:
    primeiro = buscar_trechos(pergunta_teste, k=1)[0]
    acertou = primeiro["fonte"] == fonte_esperada
    acertos += int(acertou)
    print(
        "OK" if acertou else "REVISAR",
        "| esperado:", fonte_esperada,
        "| recuperado:", primeiro["fonte"],
        "| pergunta:", pergunta_teste,
    )

print(f"\nRecall@1 didático: {acertos}/{len(casos_esperados)}")

## 15. Desafio

Escolha **uma** opção.

### A — Novos documentos

Adicione pelo menos três trechos sobre outro produto fictício. Inclua metadados e crie três perguntas de avaliação.

### B — Experimento com `k`

Compare `k=1`, `k=3` e `k=5` nas mesmas perguntas. Registre quando contexto adicional ajuda ou atrapalha.

### C — Divisão automática em chunks

Crie uma função que divida um texto maior em trechos com tamanho controlado. Preserve a fonte e compare a recuperação antes e depois.

### D — NIA completa

Combine o RAG deste laboratório com as ferramentas do Lab 3. A NIA deve consultar documentos para responder dúvidas e usar funções apenas para ações.

**Pergunta para refletir:** qual evidência é necessária para confiar em uma resposta produzida pelo seu RAG?

## 16. Espaço para sua implementação

In [ ]:
# Implemente aqui a opção A, B, C ou D do desafio.

## 17. Checklist final

- [ ] Cada documento recebeu um embedding próprio.
- [ ] A pergunta e os documentos foram preparados para recuperação.
- [ ] A similaridade foi calculada e os trechos foram ordenados.
- [ ] Os trechos recuperados ficaram visíveis para inspeção.
- [ ] A resposta usou apenas o contexto e indicou fontes.
- [ ] Um caso sem resposta na base foi testado.
- [ ] Recuperação e geração foram avaliadas separadamente.
- [ ] Uma opção do desafio foi implementada.

Você construiu um pipeline RAG completo em pequena escala: indexação, recuperação, aumento de contexto e geração fundamentada.